# BirdCLEF+ 2026 — Native Perch v2 (zero-shot)

**No training, no embeddings probe, no distillation.** Run Google's original 14,795-class Perch v2 classification head using full Perch ONNX on CPU, map exact BirdCLEF species labels, and write the 234-class `submission.csv`.

**Recommended source to Copy & Edit:** [BirdCLEF 2026 — ONNX Perch + Sequence Modeling](https://www.kaggle.com/code/vyankteshdwivedi/birdclef-2026-onnx-perch-sequence-modeling). Its Inputs list includes the **BirdCLEF+ 2026 competition**, **Perch-onnx-for-birdclef+2026**, and Google's **Perch** model (for `assets/labels.csv`). Keep those attached. Other input datasets are unnecessary for this notebook.

Kaggle settings: **Accelerator = None**, **Internet = OFF**. Copy the following cells into your fork, or import this notebook and attach the three inputs above. During an ordinary interactive Run All, Kaggle hides test audio: the notebook predicts one **training** soundscape as a smoke test and writes the public sample rows to `submission.csv`. During submission scoring, it uses every real test soundscape.

**Zero-shot coverage:** expected approximately 203/234 direct taxonomy matches. Unmapped species get a constant score (`MISSING_SCORE`); no genus proxies or learned classifier are used. Actual match count is printed and may depend on the attached Perch label version.


In [1]:
# 1. Locate competition assets and Perch ONNX + native label vocabulary.
from pathlib import Path
import sys
import subprocess
import os
import time

import numpy as np
import pandas as pd
import soundfile as sf

INPUT_ROOT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(parents=True, exist_ok=True)

CPU_THREADS = 4
BATCH_SIZE = 4                 # small batches to keep CPU RAM use predictable
MISSING_SCORE = 1e-3          # constant for species absent from Perch native head
ONNX_PATH = None              # optionally override with Path('/kaggle/input/.../perch_v2.onnx')
LABELS_PATH = None            # optionally override with Path('/kaggle/input/.../assets/labels.csv')

sample_paths = sorted(INPUT_ROOT.rglob('sample_submission.csv'))
competition_paths = [p.parent for p in sample_paths if (p.parent / 'taxonomy.csv').is_file()]
if len(competition_paths) != 1:
    raise FileNotFoundError(f'Expected one BirdCLEF 2026 competition input, found {competition_paths}. Attach BirdCLEF+ 2026.')
COMP = competition_paths[0]
print('Competition:', COMP)

candidates = [
    p for p in INPUT_ROOT.rglob('*.onnx')
    if 'perch' in str(p).lower() and 'backbone' not in str(p).lower()
]
candidates.sort(key=lambda p: (
    0 if p.name.lower() == 'perch_v2_no_dft.onnx' else
    1 if p.name.lower() == 'perch_v2.onnx' else 2,
    len(str(p)),
))
if ONNX_PATH is None:
    if not candidates:
        raise FileNotFoundError('No Perch ONNX model found. Keep/Add the Perch-onnx-for-birdclef+2026 Dataset.')
    ONNX_PATH = candidates[0]
ONNX_PATH = Path(ONNX_PATH)
print('ONNX model:', ONNX_PATH, f'({ONNX_PATH.stat().st_size / 2**20:.0f} MiB)')

label_candidates = sorted(INPUT_ROOT.rglob('labels.csv'), key=lambda p: (
    'perch' not in str(p).lower(), 'assets' not in str(p).lower(), len(str(p))
))
if LABELS_PATH is None:
    matching = []
    for path in label_candidates:
        try:
            table = pd.read_csv(path)
            if len(table) == 14795:
                matching.append(path)
        except Exception:
            pass
    if not matching:
        raise FileNotFoundError('Perch 14,795-class assets/labels.csv missing. Attach Google Perch v2 model (or a Dataset containing its labels.csv).')
    LABELS_PATH = matching[0]
LABELS_PATH = Path(LABELS_PATH)
print('Perch labels:', LABELS_PATH)

try:
    import onnxruntime as ort
except ImportError:
    wheels = sorted(INPUT_ROOT.rglob('onnxruntime*.whl'))
    if not wheels:
        raise RuntimeError('onnxruntime is missing. Attach a Kaggle Dataset with a compatible onnxruntime CPU wheel; Internet must remain OFF.')
    # pip chooses the compatible wheel from the directory. No network access.
    wheel_dirs = sorted({str(p.parent) for p in wheels})
    command = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps']
    for directory in wheel_dirs:
        command += ['--find-links', directory]
    subprocess.run(command + ['onnxruntime'], check=True)
    import onnxruntime as ort

print('ONNX Runtime:', ort.__version__)


Competition: /kaggle/input/competitions/birdclef-2026
ONNX model: /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx (390 MiB)
Perch labels: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/assets/labels.csv
Looking in links: /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026
Processing /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
ONNX Runtime: 1.24.4


In [2]:
# 2. Exact native-head class mapping: eBird code (when available), then scientific name.
sample = pd.read_csv(COMP / 'sample_submission.csv')
target_classes = sample.columns[1:].tolist()
taxonomy = pd.read_csv(COMP / 'taxonomy.csv', dtype={'primary_label': 'string'})
perch_labels = pd.read_csv(LABELS_PATH)
assert len(target_classes) == 234
assert len(perch_labels) == 14795, f'Unexpected Perch vocabulary: {len(perch_labels)}'
assert {'primary_label', 'scientific_name'}.issubset(taxonomy.columns)

normalize = lambda v: str(v).strip().casefold() if pd.notna(v) else ''
# Some label-file versions include an eBird-code field; use it only for exact matches.
code_to_idx = {}
for col in ['ebird2021', 'ebird_code', 'ebird2024', 'ebird2023']:
    if col in perch_labels:
        for i, code in perch_labels[col].items():
            if (key := normalize(code)):
                code_to_idx.setdefault(key, i)

sci_col = next((col for col in ('scientific_name', 'inat2024_fsd50k') if col in perch_labels), None)
if sci_col is None:
    raise ValueError(f'Cannot map scientific names; available Perch labels columns: {perch_labels.columns.tolist()}')
sci_to_idx = {}
for i, sci in perch_labels[sci_col].items():
    if (key := normalize(sci)):
        sci_to_idx.setdefault(key, i)

tax_by_label = taxonomy.set_index('primary_label')['scientific_name'].to_dict()
perch_index = []
match_type = []
for label in target_classes:
    code = normalize(label)
    sci = normalize(tax_by_label.get(label))
    if code in code_to_idx:
        perch_index.append(code_to_idx[code]); match_type.append('eBird code')
    elif sci in sci_to_idx:
        perch_index.append(sci_to_idx[sci]); match_type.append('scientific name')
    else:
        perch_index.append(-1); match_type.append('missing')

perch_index = np.asarray(perch_index, dtype=np.int64)
mapped_columns = np.flatnonzero(perch_index >= 0)
coverage = pd.DataFrame({'species': target_classes, 'perch_index': perch_index, 'match': match_type})
print('Native-head label coverage:', len(mapped_columns), '/', len(target_classes))
print(coverage['match'].value_counts().to_string())
print('Unmapped:', coverage.loc[coverage['match'].eq('missing'), 'species'].tolist())
coverage.to_csv(OUTPUT / 'perch-native-class-mapping.csv', index=False)

if len(mapped_columns) < 150:
    raise ValueError('Too few exact matches. Check that you attached the *Perch v2* 14,795-label vocabulary, not Perch v1.')


Native-head label coverage: 203 / 234
match
scientific name    203
missing             31
Unmapped: ['116570', '1491113', '1595929', '25073', '47158son01', '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08', '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16', '47158son17', '47158son18', '47158son19', '47158son20', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25', '516975', '74580']


In [3]:
# 3. Prepare the ONNX session. Request ONLY native classification logits (no embedding outputs).
options = ort.SessionOptions()
options.intra_op_num_threads = CPU_THREADS
options.inter_op_num_threads = 1
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session = ort.InferenceSession(str(ONNX_PATH), sess_options=options, providers=['CPUExecutionProvider'])
print('Inputs:', [(i.name, i.shape, i.type) for i in session.get_inputs()])
print('Outputs:', [(o.name, o.shape) for o in session.get_outputs()])

model_input = session.get_inputs()[0]
input_name = model_input.name
if isinstance(model_input.shape[-1], int) and model_input.shape[-1] != 160000:
    raise ValueError(f'Wrong ONNX input: expected 5s waveform (160000 samples), got {model_input.shape}')

output_names = {o.name for o in session.get_outputs()}
if 'label' not in output_names:
    raise ValueError('This is not full Perch v2: ONNX model must expose the original `label` head (14795 outputs).')
logit_shape = next(o.shape for o in session.get_outputs() if o.name == 'label')
if isinstance(logit_shape[-1], int) and logit_shape[-1] != len(perch_labels):
    raise ValueError(f'Native-head output dimension ({logit_shape[-1]}) differs from attached label vocabulary ({len(perch_labels)}).')

# A real audio model smoke test; avoids processing the hidden test during setup.
test_files = sorted((COMP / 'test_soundscapes').glob('*.ogg')) if (COMP / 'test_soundscapes').is_dir() else []
PREVIEW = not test_files
if PREVIEW:
    files = sorted((COMP / 'train_soundscapes').glob('*.ogg'))[:1]
    print('PREVIEW: hidden test audio is not visible; using exactly one training soundscape.')
else:
    files = test_files
    print('SCORING: found', len(files), 'real test soundscapes.')
if not files:
    raise FileNotFoundError('No test audio and no training soundscape for smoke test.')


Inputs: [('inputs', ['batch', 160000], 'tensor(float)')]
Outputs: [('embedding', ['batch', 1536]), ('spatial_embedding', ['batch', 16, 4, 1536]), ('spectrogram', ['batch', 500, 128]), ('label', ['batch', 14795])]
PREVIEW: hidden test audio is not visible; using exactly one training soundscape.


In [4]:
# 4. No fine-tuning: Perch native scores -> exact BirdCLEF columns -> submission.
from scipy.special import expit

SR, WINDOW_SAMPLES, N_WINDOWS = 32000, 160000, 12

def audio_windows(path):
    with sf.SoundFile(path) as sound:
        if sound.samplerate != SR:
            raise ValueError(f'Expected 32 kHz; got {sound.samplerate} in {path}')
        y = sound.read(dtype='float32', always_2d=True).mean(axis=1)
    needed = WINDOW_SAMPLES * N_WINDOWS
    if len(y) < needed:
        y = np.pad(y, (0, needed - len(y)))
    return y[:needed].reshape(N_WINDOWS, WINDOW_SAMPLES).astype(np.float32, copy=False)

def map_native_logits(logits):
    if logits.shape[1] != 14795:
        raise ValueError(f'Expected (batch, 14795) logits, received {logits.shape}')
    probs = np.full((len(logits), len(target_classes)), MISSING_SCORE, dtype=np.float32)
    probs[:, mapped_columns] = expit(logits[:, perch_index[mapped_columns]])
    return probs

all_frames = []
started = time.perf_counter()
for fi, path in enumerate(files, 1):
    waveform = audio_windows(path)
    pred = []
    for offset in range(0, N_WINDOWS, BATCH_SIZE):
        batch = waveform[offset:offset + BATCH_SIZE]
        logits = session.run(['label'], {input_name: batch})[0]
        pred.append(map_native_logits(logits))
    pred = np.concatenate(pred, axis=0)
    frame = pd.DataFrame(pred, columns=target_classes)
    frame.insert(0, 'row_id', [f'{path.stem}_{end}' for end in range(5, 61, 5)])
    all_frames.append(frame)
    if fi == 1 or fi % 25 == 0 or fi == len(files):
        elapsed = time.perf_counter() - started
        print(f'{fi}/{len(files)} soundscapes | {elapsed:.1f}s | {fi * 12 / elapsed:.2f} windows/sec', flush=True)

native_preds = pd.concat(all_frames, ignore_index=True)
assert native_preds.columns.tolist() == sample.columns.tolist()
assert native_preds.row_id.is_unique
assert np.isfinite(native_preds[target_classes].to_numpy()).all()
assert native_preds[target_classes].to_numpy().min() >= 0
assert native_preds[target_classes].to_numpy().max() <= 1

if PREVIEW:
    native_preds.to_csv(OUTPUT / 'smoke-predictions.csv', index=False)
    # BirdCLEF hides test IDs during an interactive Run All; preserve the public sample
    # for the Kaggle UI to recognize the expected output filename/format.
    sample.to_csv(OUTPUT / 'submission.csv', index=False)
    print('PREVIEW ONLY: smoke-predictions.csv contains Perch scores on real training audio.')
    print('submission.csv contains the competition sample; not a meaningful scored result.')
else:
    native_preds.to_csv(OUTPUT / 'submission.csv', index=False)
    print('READY: predicted', len(native_preds), 'test windows with native Perch v2 scores.')

print('Saved:', OUTPUT / 'submission.csv')
print('Total inference seconds:', round(time.perf_counter() - started, 1))
print('Coverage:', len(mapped_columns), '/ 234')
display(native_preds.head(3).iloc[:, :10])


1/1 soundscapes | 2.1s | 5.59 windows/sec
PREVIEW ONLY: smoke-predictions.csv contains Perch scores on real training audio.
submission.csv contains the competition sample; not a meaningful scored result.
Saved: /kaggle/working/submission.csv
Total inference seconds: 2.2
Coverage: 203 / 234


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961
0,BC2026_Train_0001_S08_20250606_030007_5,0.436654,0.001,0.200762,0.001,0.001,0.721656,0.149819,0.372669,0.085209
1,BC2026_Train_0001_S08_20250606_030007_10,0.516247,0.001,0.160439,0.001,0.001,0.741232,0.121439,0.407281,0.070889
2,BC2026_Train_0001_S08_20250606_030007_15,0.492779,0.001,0.177227,0.001,0.001,0.702288,0.133897,0.330157,0.076408


## Notes / safeguards

- This is **Perch v2 zero-shot**, not Perch embedding + MLP, ProtoSSM, or SED.
- Uses full ONNX output **`label`**, not a backbone-only model.
- Input is exactly **5 seconds, mono, 32 kHz**; predictions cover all 12 windows per minute.
- The original Perch label head is **not complete for BirdCLEF**. Unmapped classes get the same small constant score for every window and therefore have no discriminative power. Do not confuse 203-class coverage with a guarantee of accuracy.
- Class mapping is **exact only** (eBird code or scientific name), no hand-tuned genus proxies or leaderboard-specific post-processing.
- The output `perch-native-class-mapping.csv` lets you audit every mapped species.
- Online scoring happens only when Kaggle mounts **actual hidden test audio**. The public `submission.csv` generated in preview mode is a template placeholder, as in typical BirdCLEF code competitions.
- Perch ONNX CPU runtime depends on the attached model version and Kaggle CPU environment; check the log for throughput. The 90-minute competition limit applies to the full hidden test, not the 12-window preview.

**Upstream assets:** Google Perch v2 model; community ONNX export from [justinchuby/Perch-onnx](https://huggingface.co/justinchuby/Perch-onnx) / [rishikeshjani Kaggle Dataset](https://www.kaggle.com/datasets/rishikeshjani/perch-onnx-for-birdclef-2026). Original Perch model has 14,795 native classes. The original model and conversions are third-party resources, not included in this notebook file.
